In [1]:
import argparse
import io
import random
import sys
from pathlib import Path
import os
import requests
import numpy as np
from PIL import Image
import time

In [2]:
#time.sleep(600)

In [3]:
# USGS National Map — NAIP ImageServer
NAIP_EXPORT_URL = (
    "https://imagery.nationalmap.gov/arcgis/rest/services"
    "/USGSNAIPImagery/ImageServer/exportImage"
)

SPAN_DEG = 0.005  # ≈ 550 m lon, ≈ 556 m lat around Chicago Loop

MICHIGAN_BOUNDING_BOX = {
    "lon_min": -90.4181,
    "lon_max": -82.4135,
    "lat_min": 41.6961,
    "lat_max": 48.3061
}

def centre_to_bbox(lon: float, lat: float, span: float = SPAN_DEG) -> dict:
    """Return a Web Mercator bounding box dict from a WGS84 lon/lat centre."""
    return {
        "xmin": lon - span,
        "ymin": lat - span,
        "xmax": lon + span,
        "ymax": lat + span,
    }


def fetch_naip(bbox: dict, size: int = 512) -> Image.Image:
    """
    Call the USGS NAIP ImageServer exportImage endpoint and return a PIL image.

    The ImageServer accepts WGS84 (EPSG:4326) bounding boxes directly when
    bboxSR=4326 is passed.  It renders at the requested pixel size server-side,
    so no rasterio or GDAL needed.
    """
    params = {
        "bbox":         f"{bbox['xmin']},{bbox['ymin']},{bbox['xmax']},{bbox['ymax']}",
        "bboxSR":       "4326",          # input CRS: WGS84
        "size":         f"{size},{size}",
        "imageSR":      "4326",          # output CRS
        "format":       "png",
        "pixelType":    "U8",
        "noDataInterpretation": "esriNoDataMatchAny",
        "interpolation": "RSP_BilinearInterpolation",
        "renderingRule": '{"rasterFunction":"NaturalColor"}',
        "f":            "image",         # return raw image bytes
    }

    resp = requests.get(NAIP_EXPORT_URL, params=params, timeout=60)

    if resp.status_code != 200:
        raise RuntimeError(
            f"USGS API returned HTTP {resp.status_code}.\n"
            f"URL: {resp.url}\n"
            f"Body: {resp.text[:300]}"
        )

    content_type = resp.headers.get("Content-Type", "")
    if "image" not in content_type:
        raise RuntimeError(
            f"Expected image response but got: {content_type}\n"
            f"Body: {resp.text[:300]}\n"
            "Hint: the bbox may be outside NAIP coverage (continental USA only)."
        )

    return Image.open(io.BytesIO(resp.content)).convert("RGB")

def apply_gamma(img: Image.Image, gamma: float = 1.0) -> Image.Image:
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.power(np.clip(arr, 0, 1), 1.0 / gamma)
    return Image.fromarray((arr * 255).astype(np.uint8))

span = SPAN_DEG
size = 256

# Generate 1000 images
for i in range(1000):
    output = f"images/naip_rgb_{i+1}.png"

    if not os.path.exists(output):

        # Adding randomness to the coordinates
        random_lon = random.uniform(MICHIGAN_BOUNDING_BOX["lon_min"], MICHIGAN_BOUNDING_BOX["lon_max"])
        random_lat = random.uniform(MICHIGAN_BOUNDING_BOX["lat_min"], MICHIGAN_BOUNDING_BOX["lat_max"])

        if i < 196:
            continue
    
        bbox = centre_to_bbox(random_lon, random_lat, span)
        print(f"Fetching image {i+1} from bbox: {bbox}")
    
        img = fetch_naip(bbox, size=size)
        img = apply_gamma(img)
    
        if np.asarray(img).mean() < 240:
            out = Path(output)
            img.save(out)
            print(f"Saved -> {out.resolve()}")
        else:
            print(f"Skipping(all white)")
           
        if i % 15 == 0:
            time.sleep(600)

Fetching image 198 from bbox: {'xmin': -83.3791352212336, 'ymin': 47.677296499266696, 'xmax': -83.3691352212336, 'ymax': 47.6872964992667}
Skipping(all white)
Fetching image 199 from bbox: {'xmin': -89.3430430745865, 'ymin': 42.51065354033426, 'xmax': -89.33304307458651, 'ymax': 42.520653540334266}
Skipping(all white)
Fetching image 202 from bbox: {'xmin': -86.53195877220355, 'ymin': 45.96001378261055, 'xmax': -86.52195877220356, 'ymax': 45.97001378261056}
Skipping(all white)
Fetching image 203 from bbox: {'xmin': -82.85242857980508, 'ymin': 46.66490113180965, 'xmax': -82.84242857980509, 'ymax': 46.674901131809655}
Skipping(all white)
Fetching image 207 from bbox: {'xmin': -87.65765937965831, 'ymin': 46.20187483617992, 'xmax': -87.64765937965832, 'ymax': 46.21187483617992}
Skipping(all white)
Fetching image 209 from bbox: {'xmin': -86.13594477148936, 'ymin': 42.63047433432927, 'xmax': -86.12594477148937, 'ymax': 42.64047433432928}
Skipping(all white)
Fetching image 210 from bbox: {'xmi

RuntimeError: USGS API returned HTTP 502.
URL: https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPImagery/ImageServer/exportImage?bbox=-85.3466524521128%2C47.784700205556845%2C-85.33665245211282%2C47.79470020555685&bboxSR=4326&size=256%2C256&imageSR=4326&format=png&pixelType=U8&noDataInterpretation=esriNoDataMatchAny&interpolation=RSP_BilinearInterpolation&renderingRule=%7B%22rasterFunction%22%3A%22NaturalColor%22%7D&f=image
Body: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
</body>
</html>
